In [1]:
#!/usr/bin/env python

%load_ext autoreload
%autoreload 2

import ipywidgets as widgets
from ipywidgets import interact, interact_manual
from IPython.display import display
import inspect

import Dusty_Class
import AttributeStringArray
import FileWriter

In [2]:
# ipywidgets

#File Name
print("Dusty Input")
label_input_name = widgets.Label(value='Input File Name:')
input_name = widgets.Text(
                placeholder='Name File',
                disabled=False
            )
name_tb = widgets.HBox([label_input_name, input_name])
display(name_tb)


#Combination Black Body

label_bb_number = widgets.Label(value='Number of Black Bodies:')
bb_number = widgets.IntSlider(
                value=1,
                min=1,
                max=10,
                step=1,
                disabled=False,
                continuous_update=False,
                orientation='horizontal',
                readout=True,
                readout_format='d'
            )
bb_slider = widgets.HBox([label_bb_number, bb_number])

label_temps = widgets.Label(value='Temperatures (K):')
temps = widgets.Text(
    placeholder='Type numbers separated by commas (e.g. 100, 200)',
    layout=widgets.Layout(width='320px')
)
temps_tb = widgets.HBox([label_temps, temps])

#Luminosities are only important if bb number > 1
label_lums = widgets.Label(value='Fractional Luminosities:')
lums = widgets.Text(
    placeholder='Type numbers separated by commas (e.g. 100, 200)',
    layout=widgets.Layout(width='320px')
)
lums_tb = widgets.HBox([label_lums, lums])


#Broken Power Law

label_N_functions = widgets.Label(value='Number of Power Law segments:')
N_functions = widgets.BoundedIntText(
            value=1,
            min=1,
            step=1,
            disabled=False
        )
N_functions_itb = widgets.HBox([label_N_functions, N_functions])

label_lambdas = widgets.Label(value='Breakpoint Wavelengths (λ in µm)')
_lambdas = widgets.Text(
    placeholder='Type numbers separated by commas (e.g. 100, 200)',
    layout=widgets.Layout(width='320px')
)
_lambdas_tb = widgets.HBox([label_lambdas, _lambdas])

label_power_indicies = widgets.Label(value='Power Indicies (k):')
power_indicies = widgets.Text(
    placeholder='Type numbers separated by commas (e.g. 100, 200)',
    layout=widgets.Layout(width='320px')
)
power_indicies_tb = widgets.HBox([label_power_indicies, power_indicies])


#Engelke-Marengo Function

label_tbb_temp = widgets.Label(value='Temperature (K):')
tbb_temp = widgets.Text(
    layout=widgets.Layout(width='320px')
)
tbb_temp_tb = widgets.HBox([label_tbb_temp, tbb_temp])

label_sio_fd = widgets.Label(value='SiO Feature Depth (%):')
sio_fd = widgets.FloatSlider(
                value=1.0,
                min=0.1,
                max=100.0,
                step=0.1,
                disabled=False,
                continuous_update=False,
                orientation='horizontal',
                readout=True,
                readout_format='.1f'
            )
sio_fd_slider = widgets.HBox([label_sio_fd, sio_fd])


#File: lambdaFvlambda, Fvlambda, Fvv vs. lambda

label_er_files = widgets.Label(value='File Name (file must have λ in µm):')
er_files = widgets.Text(
    layout=widgets.Layout(width='320px')
)
er_files_tb = widgets.HBox([label_er_files, er_files])


cbb_box = widgets.VBox([
widgets.Label("Combination Black Body", style=dict(font_size="16px")),
bb_slider,
temps_tb,
lums_tb
])

bpl_box = widgets.VBox([
widgets.Label("Broken Power Law", style=dict(font_size="16px")),
N_functions_itb,
_lambdas_tb,
power_indicies_tb
])

emf_box = widgets.VBox([
widgets.Label("Engelke-Marengo Function", style=dict(font_size="16px")),
tbb_temp_tb,
sio_fd_slider
])

er_file_box = widgets.VBox([
widgets.Label("File: λ F_λ, F_λ, F_ν vs. λ", style=dict(font_size="16px")),
er_files_tb
])

#Spectrum
spectrum_values = [
    ('Combination Black Body', 1),
    ('Broken Power Law', 2),
    ('Engelke-Marengo Function', 3),
    ('File: λ F_λ', 4),
    ('File: F_λ', 5),
    ('File: F_ν vs. λ', 6)  
]

label_spectrum = widgets.Label(value='Spectrum Type:')
spectrum = widgets.Dropdown(
    options=spectrum_values,
    value=1,
    continuous_update=True,
    disabled=False)

# A function to observe the value change
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        print(f"Selected label: {spectrum.label}") # Access the label
        print(f"Selected value: {change['new']}") # Access the value
        if len(file) > 1:
            file[1].spectrum = change['new']

# Observe the dropdown for changes
spectrum.observe(on_change, names='value')

er_section = widgets.VBox([], continous_update=True)

def visibility(change):
#        if change['type'] == 'change' and change['name'] == 'value':
            value = change['new']
            er_section.children = []
            if value == 1:
                er_section.children = [cbb_box]
            elif value == 2:
                er_section.children = [bpl_box]
            elif value == 3:
                er_section.children = [emf_box]
            elif value in [4,5,6]:
                er_section.children = [er_file_box]
            print("dropdown changed")

spectrum.observe(visibility, names='value')

visibility({'new': spectrum.value})

spectrum_dd = widgets.HBox([label_spectrum, spectrum])

parameters_directory = widgets.Tab()

er_box = widgets.VBox([widgets.Label("External Radiation", style=dict(font_size="22px")), 
                       spectrum_dd,
                       er_section
])


dp_box = widgets.VBox([widgets.Label("Dust Properties Content")])
dd_box = widgets.VBox([widgets.Label("Density Distribution Content")])
od_box = widgets.VBox([widgets.Label("Optical Depth Content")])

parameters_directory.children = [er_box, dp_box, dd_box, od_box]

titles = ["External Radiation", "Dust Properties", "Density Distribution", "Optical Depth"]
for x, title in enumerate(titles):
    parameters_directory.set_title(x, title)

display(parameters_directory)

Dusty Input


dropdown changed


In [3]:
import ipywidgets as widgets
from IPython.display import display

spectrum = widgets.Dropdown(
    options=[("CBB",1),("BPL",2),("EMF",3)],
    value=1
)

cbb_box = widgets.VBox([widgets.Label("CBB settings")])
bpl_box = widgets.VBox([widgets.Label("BPL settings")])
emf_box = widgets.VBox([widgets.Label("EMF settings")])

er_section = widgets.VBox()

def visibility(change):
    value = change['new']

    if value == 1:
        er_section.children = [cbb_box]
    elif value == 2:
        er_section.children = [bpl_box]
    elif value == 3:
        er_section.children = [emf_box]

spectrum.observe(visibility, names='value')

visibility({'new': spectrum.value})

display(widgets.VBox([spectrum, er_section]))